# 📦 Phase 2 — Image Preprocessing

**Project:** Facial Skin Condition Detection  
**Level:** MCA Academic Project  

---

## Overview

This notebook demonstrates the image preprocessing pipeline implemented in `src/preprocessing.py`.

### Pipeline Steps
1. **Resize** — All images are resized to **128 × 128 pixels** for uniform input
2. **CLAHE** — Contrast Limited Adaptive Histogram Equalisation applied to the L* channel (LAB colour space) for illumination normalisation
3. **Gaussian Blur** — Mild noise reduction using a 3×3 Gaussian kernel

> ⚠️ **Why not use raw images?** Skin images vary widely in brightness, camera settings, and noise levels. Preprocessing ensures the feature extractor operates on normalised, consistent input.

In [ ]:
import sys
from pathlib import Path

# Add src/ to path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from preprocessing import load_and_preprocess_image, preprocess_pipeline, TARGET_SIZE

print(f'Target size      : {TARGET_SIZE}')
print(f'Project root     : {PROJECT_ROOT}')

## 1. Visualise Preprocessing on Sample Images

For each class, we load the first available image and display **Before** vs **After** preprocessing.

In [ ]:
CLASS_FOLDERS = {
    'Acne':                  'Acne',
    'Rosacea':               'Rosacea',
    'Melasma':               'Melasma',
    'Facial Eczema':         'Facial_Eczema',
    'Seborrheic Dermatitis': 'Seborrheic_Dermatitis',
}
SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
DATASET_DIR   = PROJECT_ROOT / 'dataset'

samples = {}
for cls_name, folder in CLASS_FOLDERS.items():
    folder_path = DATASET_DIR / folder
    if folder_path.exists():
        imgs = [f for f in folder_path.rglob('*') if f.suffix.lower() in SUPPORTED_EXT]
        if imgs:
            samples[cls_name] = imgs[0]

print(f'Found sample images for {len(samples)} class(es):')
for k, v in samples.items():
    print(f'  {k:<28} → {v.name}')

In [ ]:
if not samples:
    print('⚠ No images found. Add images to dataset/ folders first.')
else:
    n = len(samples)
    fig, axes = plt.subplots(n, 2, figsize=(10, 3.5 * n))
    if n == 1:
        axes = [axes]
    fig.suptitle('Preprocessing: Before vs After', fontsize=15, fontweight='bold', y=1.01)

    for row, (cls_name, img_path) in enumerate(samples.items()):
        raw  = cv2.imread(str(img_path))
        proc = preprocess_pipeline(raw)

        raw_rgb  = cv2.cvtColor(raw,  cv2.COLOR_BGR2RGB)
        proc_rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)

        axes[row][0].imshow(raw_rgb)
        axes[row][0].set_title(f'{cls_name} — Original ({raw.shape[1]}×{raw.shape[0]})', fontsize=10)
        axes[row][0].axis('off')

        axes[row][1].imshow(proc_rgb)
        axes[row][1].set_title(f'{cls_name} — Preprocessed ({proc.shape[1]}×{proc.shape[0]})', fontsize=10)
        axes[row][1].axis('off')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'preprocessing_samples.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved → reports/preprocessing_samples.png')

## 2. Step-by-Step Pipeline Visualisation

Below we show each individual step applied to a single image so the effect of each operation is clear.

In [ ]:
if samples:
    sample_path = next(iter(samples.values()))
    img_bgr     = cv2.imread(str(sample_path))

    # Step 1: Resize
    step1 = cv2.resize(img_bgr, TARGET_SIZE, interpolation=cv2.INTER_AREA)

    # Step 2: CLAHE
    lab = cv2.cvtColor(step1, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    step2 = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    # Step 3: Gaussian blur
    step3 = cv2.GaussianBlur(step2, (3, 3), 0)

    stages = [
        ('Original',        img_bgr),
        ('1. Resized 128×128', step1),
        ('2. CLAHE',        step2),
        ('3. Gaussian Blur', step3),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, (title, img) in zip(axes, stages):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.axis('off')
    plt.suptitle('Preprocessing Pipeline Steps', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Smoke Test — All Classes

In [ ]:
import importlib, preprocessing
importlib.reload(preprocessing)
from preprocessing import _demo
_demo()

---
## Summary

| Step | Operation | Effect |
|------|-----------|--------|
| 1 | Resize to 128×128 | Uniform input size for all algorithms |
| 2 | CLAHE (LAB L* channel) | Illumination normalisation; enhances local contrast |
| 3 | Gaussian Blur (3×3) | Reduces high-frequency noise |

**Next:** Run `03_feature_extraction.ipynb` to extract the 58-dimensional feature vector.